## Universe 100 — RL-label EDA, modelling, and diagnostics

## Cleaned notebook changes

- Each regression, binary, and multiclass model is fitted only once per target.
- F1 and Balanced Accuracy thresholds are selected from validation probabilities without retraining.
- Probability, calibration, Top-K, year, industry, ticker, and stability diagnostics share one helper system.
- The previous duplicated helper functions, repeated threshold-training cells, and monolithic final diagnostic cell were removed.
- Threshold-dependent results are stored separately from genuinely distinct fitted-model results.


## 1. Imports and paths

In [ ]:
%pip install lightgbm

: 

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, pearsonr

from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    classification_report,
)

RANDOM_STATE = 42

# Optional models
HAS_LGBM = False
HAS_XGB = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier

    HAS_LGBM = True
    print("LightGBM available.")
except ImportError:
    print("LightGBM not installed. Skipping LightGBM models.")

try:
    from xgboost import XGBRegressor, XGBClassifier

    HAS_XGB = True
    print("XGBoost available.")
except ImportError:
    print("XGBoost not installed. Skipping XGBoost models.")

In [ ]:
## Load dataset
# jsonl input paths
train_path = Path(r"C:\Users\user\Downloads\universe_100_full_post_normalisation_train.jsonl\universe_100_full_post_normalisation_train.jsonl")
validation_path = Path(r"C:\Users\user\Downloads\universe_100_full_post_normalisation_validation.jsonl\universe_100_full_post_normalisation_validation.jsonl")
test_path = Path(r"C:\Users\user\Downloads\universe_100_full_post_normalisation_test.jsonl\universe_100_full_post_normalisation_test.jsonl")

# set output directory
output_dir = Path(r"C:\Users\user\Downloads\universe_100_full_post_normalisation_0527")
output_dir.mkdir(parents=True, exist_ok=True)
# parquet output paths
train_parquet_path = output_dir / "universe_100_full_post_normalisation_train.parquet"
validation_parquet_path = output_dir / "universe_100_full_post_normalisation_validation.parquet"
test_parquet_path = output_dir / "universe_100_full_post_normalisation_test.parquet"

# column list output paths
feature_cols_path = output_dir / "feature_columns.txt"
attribute_cols_path = output_dir / "attribute_columns.txt"
label_cols_path = output_dir / "label_columns.txt"

print("Train JSONL:", train_path)
print("Validation JSONL:", validation_path)
print("Test JSONL:", test_path)

print("\nOutput directory:", output_dir)
print("Train parquet:", train_parquet_path)
print("Validation parquet:", validation_parquet_path)
print("Test parquet:", test_parquet_path)
print("Feature columns:", feature_cols_path)
print("Attribute columns:", attribute_cols_path)
print("Label columns:", label_cols_path)

## 2. Load / flatten / prepare data

In [ ]:
# Scan JSONL before flattening
n_rows = 0

ticker_counter = Counter()
date_counter = Counter()
feature_key_counter = Counter()
label_key_counter = Counter()
attribute_key_counter = Counter()

feature_count_per_row = Counter()
label_count_per_row = Counter()
attribute_count_per_row = Counter()

ticker_first_date = {}
ticker_last_date = {}

min_ts = None
max_ts = None

with open(train_path, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        if record.get("section") != "data":
            continue

        data = record.get("data", {})
        attrs = data.get("Attributes", {})
        feats = data.get("Features", {})
        labs = data.get("Labels", {})

        ticker = attrs.get("ticker")
        timestamp = attrs.get("timestamp")
        date = timestamp[:10] if timestamp else None

        n_rows += 1

        if ticker:
            ticker_counter.update([ticker])

        if date:
            date_counter.update([date])

            if ticker:
                ticker_first_date[ticker] = min(ticker_first_date.get(ticker, date), date)
                ticker_last_date[ticker] = max(ticker_last_date.get(ticker, date), date)

            min_ts = min(min_ts, timestamp) if min_ts else timestamp
            max_ts = max(max_ts, timestamp) if max_ts else timestamp

        attribute_key_counter.update(attrs.keys())
        feature_key_counter.update(feats.keys())
        label_key_counter.update(labs.keys())

        attribute_count_per_row.update([len(attrs)])
        feature_count_per_row.update([len(feats)])
        label_count_per_row.update([len(labs)])



print("\nRaw JSONL Scan Summary")
print("Rows scanned:", n_rows)
print("Unique tickers:", len(ticker_counter))
print("Unique dates:", len(date_counter))
print("Date range:", min_ts, "to", max_ts)
print("Unique attribute keys:", len(attribute_key_counter))
print("Unique feature keys:", len(feature_key_counter))
print("Unique label keys:", len(label_key_counter))

print("\nKeys per row")
print("Attribute counts per row:", attribute_count_per_row)
print("Feature counts per row:", feature_count_per_row)
print("Label counts per row:", label_count_per_row)

if date_counter:
    date_counts = list(date_counter.values())
    print("\n Rows per date ")
    print("Min rows per date:", min(date_counts))
    print("Max rows per date:", max(date_counts))
    print("Average rows per date:", sum(date_counts) / len(date_counts))


In [ ]:
def load_flatten_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)

            if record.get("section") != "data":
                continue

            data = record.get("data", {})
            attrs = data.get("Attributes", {})
            feats = data.get("Features", {})
            labs = data.get("Labels", {})

            row = {
                "IndexReference": data.get("IndexReference")
            }

            row.update({f"attr__{k}": v for k, v in attrs.items()})
            row.update({f"feature__{k}": v for k, v in feats.items()})
            row.update({f"label__{k}": v for k, v in labs.items()})

            rows.append(row)

    return pd.DataFrame(rows)

df_train = load_flatten_jsonl(train_path)
df_valid = load_flatten_jsonl(validation_path)
df_test = load_flatten_jsonl(test_path)

print("Shape:", df_train.shape)
print("Shape:", df_valid.shape)
print("Shape:", df_test.shape)

In [ ]:
def prepare_flattened_df(df):
    attribute_cols = [c for c in df.columns if c.startswith("attr__")]
    feature_cols = [c for c in df.columns if c.startswith("feature__")]
    label_cols = [c for c in df.columns if c.startswith("label__")]

    # convert timestamp to datetime
    if "attr__timestamp" in df.columns:
        df["attr__timestamp"] = pd.to_datetime(df["attr__timestamp"], errors="coerce")

    categorical_attr_cols = [
        "attr__ticker",
        "attr__sic2",
        "attr__sic_code",
        "attr__sic_description",
        "attr__regime_inflation",
        "attr__regime_vix",
        "attr__regime_yield_curve",
        "attr__daily_timeframe",
        "attr__report_timeframe",
        "attr__daily_balance_timeframe",
        "attr__fiscal_period",
    ]

    for col in categorical_attr_cols:
        if col in df.columns:
            df[col] = df[col].astype("category")

    for col in feature_cols + label_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    return df, attribute_cols, feature_cols, label_cols

df_train, train_attribute_cols, train_feature_cols, train_label_cols = prepare_flattened_df(df_train)
df_valid, valid_attribute_cols, valid_feature_cols, valid_label_cols = prepare_flattened_df(df_valid)
df_test, test_attribute_cols, test_feature_cols, test_label_cols = prepare_flattened_df(df_test)

print("Flattened Dataset Comparison")

comparison = pd.DataFrame({
    "train": [df_train.shape[0], df_train.shape[1], len(train_attribute_cols), len(train_feature_cols), len(train_label_cols), 
    df_train["attr__ticker"].nunique(), df_train["attr__timestamp"].min(), df_train["attr__timestamp"].max(), df_train.memory_usage(deep=True).sum() / 1024**3],
    "valid": [df_valid.shape[0], df_valid.shape[1], len(valid_attribute_cols), len(valid_feature_cols), len(valid_label_cols),
    df_valid["attr__ticker"].nunique(), df_valid["attr__timestamp"].min(), df_valid["attr__timestamp"].max(), df_valid.memory_usage(deep=True).sum() / 1024**3],
    "test": [df_test.shape[0], df_test.shape[1], len(test_attribute_cols), len(test_feature_cols), len(test_label_cols),
    df_test["attr__ticker"].nunique(), df_test["attr__timestamp"].min(), df_test["attr__timestamp"].max(), df_test.memory_usage(deep=True).sum() / 1024**3]}, 
    index=["rows", "columns", "attribute_cols", "feature_cols", "label_cols", "ticker_count", "start_date", "end_date", "memory_gb"])
display(comparison)

print("\nColumn Consistency across splits")
print("Tickers consistent across splits: ", set(df_train["attr__ticker"]) == set(df_valid["attr__ticker"]) == set(df_test["attr__ticker"]))
print("Feature columns consistent across splits:", set(train_feature_cols) == set(valid_feature_cols) == set(test_feature_cols))
print("Label columns consistent across splits:", set(train_label_cols) == set(valid_label_cols) == set(test_label_cols))

In [ ]:
# Save column inventories 
with open(attribute_cols_path, "w", encoding="utf-8") as f:
    for col in train_attribute_cols:
        f.write(col + "\n")

with open(feature_cols_path, "w", encoding="utf-8") as f:
    for col in train_feature_cols:
        f.write(col + "\n")

with open(label_cols_path, "w", encoding="utf-8") as f:
    for col in train_label_cols:
        f.write(col + "\n")

print("\nSaved attribute columns to:", attribute_cols_path)
print("Saved feature columns to:", feature_cols_path)
print("Saved label columns to:", label_cols_path)

# Save parquet 
df_train.to_parquet(train_parquet_path, index=False)
print("Saved flattened parquet to:", train_parquet_path)
df_valid.to_parquet(validation_parquet_path, index=False)
print("Saved flattened parquet to:", validation_parquet_path)
df_test.to_parquet(test_parquet_path, index=False)
print("Saved flattened parquet to:", test_parquet_path)

In [ ]:
attribute_cols = train_attribute_cols
all_feature_cols = train_feature_cols
label_cols = train_label_cols
primary_target = "label__pi_hindsight_entry_long"

## 3. Feature filtering & EDA

    3.1 remove missing features
    3.2 remove highly correlated features
    3.3 rank features by feature-target correlation
    3.4 define final feature set
Feature Set A：Basic cleaned features

In [ ]:
print("Original feature count:", len(all_feature_cols))

# Remove high-missing, constant, near-empty features
MISSING_THRESHOLD = 0.50
feature_quality = []

for col in all_feature_cols:
    s = df_train[col]
    feature_quality.append({
        "column": col,
        "missing_rate": s.isna().mean(),
        "n_unique": s.nunique(dropna=True),
        "std": s.std(skipna=True),
        "zero_rate": (s == 0).mean(),
    })

feature_quality = pd.DataFrame(feature_quality)

filtered_features = feature_quality.loc[
    (feature_quality["missing_rate"] <= MISSING_THRESHOLD) &
    (feature_quality["n_unique"] > 1) &
    (feature_quality["std"].fillna(0) > 0),
    "column"
].tolist()

print("After missing / constant filter:", len(filtered_features))

display(
    feature_quality.sort_values("missing_rate", ascending=False).head(15)
)

In [ ]:

# Remove highly redundant features using train set only
CORR_THRESHOLD = 0.95

# Use Spearman because many financial features are non-normal / rank-like
corr_matrix = df_train[filtered_features].corr(method="spearman").abs()

# Use upper triangle to avoid duplicate pairs
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop_corr = [
    column for column in upper.columns
    if any(upper[column] > CORR_THRESHOLD)
]

decorrelated_features = [
    c for c in filtered_features
    if c not in to_drop_corr
]
basic_cleaned_features = decorrelated_features.copy()

print("Before correlation filter:", len(filtered_features))
print("Dropped highly correlated features:", len(to_drop_corr))
print("After correlation filter:", len(decorrelated_features))
print("Basic cleaned feature count:", len(basic_cleaned_features))

# save inventories
pd.Series(basic_cleaned_features).to_csv(
    output_dir / "basic_cleaned_features.txt",
    index=False,
    header=False
)


### Feature from project B

In [ ]:
feature_file = Path("C:/Users/user/Downloads/universe_100_full_post_normalisation_0527/feature_project_b.txt")
with open(feature_file, "r", encoding="utf-8") as f:
    raw_project_b_features = [
        line.strip()
        for line in f
        if line.strip()
    ]

project_b_features_prefixed = [
    f"feature__{c}" if not c.startswith("feature__") else c
    for c in raw_project_b_features
]
##### temperary
project_b_features_without_b = [
    c for c in project_b_features_prefixed
    if not c.startswith("feature__b_")
]


common_columns = (
    set(df_train.columns)
    & set(df_valid.columns)
    & set(df_test.columns)
)

project_b_features = [
    c for c in project_b_features_without_b
    if c in common_columns
]

project_b_features_missing = [
    c for c in project_b_features_without_b
    if c not in common_columns
]

# Optional: identify which split is missing each feature
project_b_missing_details = {
    c: {
        "missing_train": c not in df_train.columns,
        "missing_valid": c not in df_valid.columns,
        "missing_test": c not in df_test.columns,
    }
    for c in project_b_features_missing
}

print("Number of raw Project B features:", len(raw_project_b_features))
print(
    "Number after temporarily excluding feature__b_:",
    len(project_b_features_without_b)
)
print(
    "Available in train, validation and test:",
    len(project_b_features)
)
print(
    "Missing from at least one dataset:",
    len(project_b_features_missing)
)

print("\nFirst 50 missing features:")
print(project_b_features_missing[:50])

print(
    "\nFirst-trial Project B model features:",
    len(project_b_features)
)


### Feature EDA

In [ ]:
# EDA on decorrelated features
def summarise_columns(data, cols):
    summary = []

    for col in cols:
        s = data[col]
        summary.append({
            "column": col,
            "dtype": str(s.dtype),
            "count": s.count(),
            "missing_count": s.isna().sum(),
            "missing_rate": s.isna().mean(),
            "zero_count": (s == 0).sum(),
            "zero_rate": (s == 0).mean(),
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "p01": s.quantile(0.01),
            "p05": s.quantile(0.05),
            "p25": s.quantile(0.25),
            "median": s.quantile(0.50),
            "p75": s.quantile(0.75),
            "p95": s.quantile(0.95),
            "p99": s.quantile(0.99),
            "max": s.max(),
            "n_unique": s.nunique(dropna=True),
        })

    return pd.DataFrame(summary)

decorrelated_feature_summary = summarise_columns(df_train, decorrelated_features)

decorrelated_feature_summary.to_csv(
    output_dir / "decorrelated_feature_summary.csv",
    index=False
)

print("Decorrelated feature summary:")
display(decorrelated_feature_summary)

### Plotting feature (later)

In [ ]:
# # Select features for plotting
# PLOT_FEATURE_N = 30

# plot_features = (
#     decorrelated_feature_summary
#     .sort_values(["missing_rate", "std"], ascending=[True, False])
#     .head(PLOT_FEATURE_N)["column"]
#     .tolist()
# )

# print("Plotting features:", len(plot_features))

# for col in plot_features:
#     s = df_train[col].dropna()

#     plt.figure(figsize=(8, 5))
#     plt.hist(s, bins=50)
#     plt.title(f"Distribution: {col}")
#     plt.xlabel(col)
#     plt.ylabel("Frequency")
#     plt.tight_layout()

#     safe_name = (
#         col.replace("feature__", "")
#         .replace("/", "_")
#         .replace(".", "_")
#         .replace(" ", "_")
#     )

#     plt.savefig(output_dir / f"hist_{safe_name}.png", dpi=150)
#     plt.show()

## 4. RL label framework and target selection

### Interpretation carried forward from the Go evaluator

The evaluator creates **RL-style supervised labels**, rather than training a reinforcement-learning agent inside this notebook.

- `trainable` / `mask` fields are **eligibility filters**, not prediction targets.
- `action_label` is the natural **classification target** (for example long vs no-trade, or short vs no-trade).
- `action_quality` is the preferred **continuous/ranking target**, because it represents opportunity quality before combining every execution/cost term.
- `reward_trade` is a useful secondary **regression target**, but it mixes opportunity, execution quality and action cost.
- component rewards are mainly diagnostic or auxiliary targets.

The code below discovers the exact RL columns available in this export instead of assuming one naming convention.

In [ ]:
# Discover RL-related labels across all splits
common_label_cols = sorted(set(train_label_cols) & set(valid_label_cols) & set(test_label_cols))
rl_label_cols = [
    c for c in common_label_cols
    if any(token in c.lower() for token in ["rl", "reward", "action_quality", "action_label", "trainable"])
]

print(f"Common label columns: {len(common_label_cols)}")
print(f"RL-related label columns: {len(rl_label_cols)}")
for c in rl_label_cols:
    print(c)

# Display compact metadata to guide target choice
def label_inventory(df, cols):
    rows = []
    for c in cols:
        s = df[c]
        non_null = s.dropna()
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "non_null_n": int(non_null.shape[0]),
            "missing_rate": float(s.isna().mean()),
            "n_unique": int(non_null.nunique()),
            "min": float(non_null.min()) if len(non_null) and pd.api.types.is_numeric_dtype(non_null) else np.nan,
            "max": float(non_null.max()) if len(non_null) and pd.api.types.is_numeric_dtype(non_null) else np.nan,
            "mean": float(non_null.mean()) if len(non_null) and pd.api.types.is_numeric_dtype(non_null) else np.nan,
            "std": float(non_null.std()) if len(non_null) and pd.api.types.is_numeric_dtype(non_null) else np.nan,
        })
    return pd.DataFrame(rows)

rl_inventory_train = label_inventory(df_train, rl_label_cols)
display(rl_inventory_train)

In [ ]:
# Flexible resolver: scores candidate columns by semantic tokens.
def resolve_label(columns, include_all=(), include_any=(), exclude=()):
    matches = []
    for c in columns:
        name = c.lower()
        if include_all and not all(tok in name for tok in include_all):
            continue
        if include_any and not any(tok in name for tok in include_any):
            continue
        if any(tok in name for tok in exclude):
            continue
        matches.append(c)
    return matches

RL_CANDIDATES = {
    "trainable": resolve_label(rl_label_cols, include_any=("trainable",)),
    "action_label": resolve_label(rl_label_cols, include_all=("action", "label")),
    "action_quality": resolve_label(rl_label_cols, include_all=("action", "quality")),
    "reward_trade": resolve_label(rl_label_cols, include_all=("reward", "trade")),
    "reward_no_action": resolve_label(rl_label_cols, include_all=("reward", "no", "action")),
    "reward_quality_component": resolve_label(rl_label_cols, include_all=("reward", "quality")),
}

for family, cols in RL_CANDIDATES.items():
    print(f"\n{family}: {cols}")

# Prefer long-oriented columns when both long and short variants exist.
def prefer_direction(cols, direction="long"):
    directed = [c for c in cols if direction in c.lower()]
    return directed[0] if directed else (cols[0] if cols else None)

RL_TRAINABLE_COL = prefer_direction(RL_CANDIDATES["trainable"], "long")
RL_ACTION_LABEL_COL = prefer_direction(RL_CANDIDATES["action_label"], "long")
RL_ACTION_QUALITY_COL = prefer_direction(RL_CANDIDATES["action_quality"], "long")
RL_REWARD_TRADE_COL = prefer_direction(RL_CANDIDATES["reward_trade"], "long")

print("\nResolved primary columns")
print("Trainable:", RL_TRAINABLE_COL)
print("Action label:", RL_ACTION_LABEL_COL)
print("Action quality:", RL_ACTION_QUALITY_COL)
print("Reward trade:", RL_REWARD_TRADE_COL)

### 4.1 Build trainable masks and model targets

The notebook creates three complementary tasks:

1. **Primary regression/ranking:** `action_quality`.
2. **Secondary regression:** `reward_trade`.
3. **Primary classification:** trade action versus no-trade, derived from `action_label` where available; otherwise from the evaluator rule `reward_trade > 0.50`.

Rows marked non-trainable or masked are excluded from modelling and reported separately.

In [ ]:
MASK_TOKENS = {"mask", "masked", "invalid", "false", "0"}
NO_TRADE_TOKENS = {"no_trade", "no trade", "none", "hold", "flat", "0"}

def make_trainable_mask(df):
    if RL_TRAINABLE_COL and RL_TRAINABLE_COL in df.columns:
        s = df[RL_TRAINABLE_COL]
        if pd.api.types.is_numeric_dtype(s):
            return s.fillna(0).astype(float).gt(0)
        return ~s.astype(str).str.strip().str.lower().isin(MASK_TOKENS)
    if RL_ACTION_LABEL_COL and RL_ACTION_LABEL_COL in df.columns:
        return ~df[RL_ACTION_LABEL_COL].astype(str).str.strip().str.lower().isin({"mask", "masked"})
    return pd.Series(True, index=df.index)


def add_rl_targets(df):
    df = df.copy()
    df["rl__trainable_mask"] = make_trainable_mask(df)

    # Preserve continuous targets under stable notebook names.
    if RL_ACTION_QUALITY_COL:
        df["target__rl_action_quality"] = pd.to_numeric(df[RL_ACTION_QUALITY_COL], errors="coerce")
    if RL_REWARD_TRADE_COL:
        df["target__rl_reward_trade"] = pd.to_numeric(df[RL_REWARD_TRADE_COL], errors="coerce")

    # Binary action target: 1 = take the evaluated trade direction, 0 = no trade.
    if RL_ACTION_LABEL_COL:
        raw = df[RL_ACTION_LABEL_COL].astype(str).str.strip().str.lower()
        df["target__rl_trade_binary"] = (~raw.isin(NO_TRADE_TOKENS | {"mask", "masked", "nan"})).astype(float)
        df.loc[raw.isin({"mask", "masked", "nan"}), "target__rl_trade_binary"] = np.nan
    elif RL_REWARD_TRADE_COL:
        reward = pd.to_numeric(df[RL_REWARD_TRADE_COL], errors="coerce")
        df["target__rl_trade_binary"] = (reward > 0.50).astype(float)
        df.loc[reward.isna(), "target__rl_trade_binary"] = np.nan
    else:
        df["target__rl_trade_binary"] = np.nan

    # Optional 3-class action target when a categorical action label is actually present.
    if RL_ACTION_LABEL_COL:
        raw = df[RL_ACTION_LABEL_COL].astype(str).str.strip().str.lower()
        mapping = {
            "no_trade": 0, "no trade": 0, "none": 0, "hold": 0, "flat": 0, "0": 0,
            "long": 1, "buy": 1, "1": 1,
            "short": 2, "sell": 2, "-1": 2,
        }
        df["target__rl_action_3class"] = raw.map(mapping).astype(float)
    return df


df_train = add_rl_targets(df_train)
df_valid = add_rl_targets(df_valid)
df_test = add_rl_targets(df_test)

RL_MODEL_TARGETS = [c for c in [
    "target__rl_action_quality",
    "target__rl_reward_trade",
    "target__rl_trade_binary",
    "target__rl_action_3class",
] if c in df_train.columns and df_train[c].notna().any()]

print("Available model targets:", RL_MODEL_TARGETS)
for split_name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    print(split_name, "trainable rate:", round(df["rl__trainable_mask"].mean(), 4))

## 5. RL target EDA

In [ ]:
def target_summary_by_split(target_col):
    rows = []
    for split_name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
        mask = df["rl__trainable_mask"] & df[target_col].notna()
        s = df.loc[mask, target_col]
        row = {
            "split": split_name,
            "target": target_col,
            "n": len(s),
            "missing_or_masked_rate": 1 - len(s) / max(len(df), 1),
            "n_unique": s.nunique(),
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "p01": s.quantile(.01),
            "p10": s.quantile(.10),
            "median": s.median(),
            "p90": s.quantile(.90),
            "p99": s.quantile(.99),
            "max": s.max(),
            "zero_rate": (s == 0).mean(),
        }
        rows.append(row)
    return pd.DataFrame(rows)

rl_target_summaries = pd.concat(
    [target_summary_by_split(t) for t in RL_MODEL_TARGETS],
    ignore_index=True,
)
display(rl_target_summaries)

In [ ]:
# Distribution plots and class shares
for target_col in RL_MODEL_TARGETS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, (split_name, df) in zip(axes, [("train", df_train), ("valid", df_valid), ("test", df_test)]):
        s = df.loc[df["rl__trainable_mask"], target_col].dropna()
        if s.nunique() <= 10:
            s.value_counts(normalize=True).sort_index().plot(kind="bar", ax=ax)
            ax.set_ylabel("Share")
        else:
            ax.hist(s, bins=60)
            ax.set_ylabel("Count")
        ax.set_title(f"{target_col}\n{split_name}")
        ax.set_xlabel("Target value")
    plt.tight_layout()
    plt.show()

In [ ]:
TIMESTAMP_COL = "attr__timestamp"
TICKER_COL = "attr__ticker"
INDUSTRY_CANDIDATES = ["attr__sic_description", "attr__sic2", "attr__sic_code"]
INDUSTRY_COL = next((c for c in INDUSTRY_CANDIDATES if c in df_train.columns), None)


def grouped_target_summary(df, target_col, group_col, min_n=20):
    work = df.loc[df["rl__trainable_mask"] & df[target_col].notna(), [group_col, target_col]].copy()
    out = work.groupby(group_col, observed=True)[target_col].agg(["count", "mean", "std", "median", "min", "max"]).reset_index()
    out["zero_rate"] = work.groupby(group_col, observed=True)[target_col].apply(lambda s: (s == 0).mean()).values
    return out[out["count"] >= min_n].sort_values("count", ascending=False)

rl_target_by_year = {}
rl_target_by_ticker = {}
rl_target_by_industry = {}

for target_col in RL_MODEL_TARGETS:
    for df in (df_train, df_valid, df_test):
        if TIMESTAMP_COL in df.columns:
            df["attr__year"] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce").dt.year

    rl_target_by_year[target_col] = pd.concat([
        grouped_target_summary(df, target_col, "attr__year", min_n=1).assign(split=split_name)
        for split_name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]
    ], ignore_index=True)

    rl_target_by_ticker[target_col] = pd.concat([
        grouped_target_summary(df, target_col, TICKER_COL, min_n=20).assign(split=split_name)
        for split_name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]
    ], ignore_index=True)

    if INDUSTRY_COL:
        rl_target_by_industry[target_col] = pd.concat([
            grouped_target_summary(df, target_col, INDUSTRY_COL, min_n=20).assign(split=split_name)
            for split_name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]
        ], ignore_index=True)

print("Industry column:", INDUSTRY_COL)
for target_col in RL_MODEL_TARGETS:
    print("\nYEAR:", target_col)
    display(rl_target_by_year[target_col])

## 6. Modelling utilities

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss


def get_model_frame(df, feature_cols, target_col):
    keep = df["rl__trainable_mask"] & df[target_col].notna()
    X = df.loc[keep, feature_cols].replace([np.inf, -np.inf], np.nan).astype(np.float32)
    y = df.loc[keep, target_col]
    meta_cols = [c for c in [TIMESTAMP_COL, TICKER_COL, INDUSTRY_COL] if c and c in df.columns]
    meta = df.loc[keep, meta_cols].copy()
    return X, y, meta


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    pearson = pearsonr(y_true, y_pred)[0] if len(np.unique(y_true)) > 1 and len(np.unique(y_pred)) > 1 else np.nan
    spearman = spearmanr(y_true, y_pred).correlation if len(np.unique(y_true)) > 1 and len(np.unique(y_pred)) > 1 else np.nan
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "Pearson": pearson,
        "Spearman": spearman,
    }


def binary_metrics(y_true, y_pred, y_score):
    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": np.nan,
        "PR_AUC": np.nan,
        "Brier": np.nan,
    }
    if len(np.unique(y_true)) == 2:
        out["ROC_AUC"] = roc_auc_score(y_true, y_score)
        out["PR_AUC"] = average_precision_score(y_true, y_score)
        out["Brier"] = brier_score_loss(y_true, y_score)
    return out


def multiclass_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted_F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def get_positive_proba(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        return p[:, 1] if p.shape[1] > 1 else np.zeros(len(X))
    if hasattr(model, "decision_function"):
        z = model.decision_function(X)
        return 1 / (1 + np.exp(-z))
    return model.predict(X).astype(float)

In [ ]:
def get_regression_models():
    models = {
        "DummyMean": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", DummyRegressor(strategy="mean"))]),
        "ElasticNet": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=RANDOM_STATE))]),
        "RandomForest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=20, n_jobs=-1, random_state=RANDOM_STATE))]),
    }
    if HAS_LGBM:
        models["LightGBM"] = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", LGBMRegressor(n_estimators=500, learning_rate=.03, num_leaves=31, min_child_samples=50, subsample=.8, colsample_bytree=.8, random_state=RANDOM_STATE, n_jobs=-1))])
    if HAS_XGB:
        models["XGBoost"] = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBRegressor(n_estimators=500, learning_rate=.03, max_depth=4, subsample=.8, colsample_bytree=.8, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1))])
    return models


def get_binary_models(y_train):
    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    spw = neg / max(pos, 1)
    models = {
        "DummyMostFrequent": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", DummyClassifier(strategy="most_frequent"))]),
        "LogisticRegression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))]),
        "RandomForest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestClassifier(n_estimators=250, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))]),
    }
    if HAS_LGBM:
        models["LightGBM"] = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", LGBMClassifier(n_estimators=500, learning_rate=.03, num_leaves=31, min_child_samples=50, subsample=.8, colsample_bytree=.8, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))])
    if HAS_XGB:
        models["XGBoost"] = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBClassifier(n_estimators=500, learning_rate=.03, max_depth=4, subsample=.8, colsample_bytree=.8, objective="binary:logistic", eval_metric="logloss", tree_method="hist", scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1))])
    return models


def get_multiclass_models():
    models = {
        "DummyMostFrequent": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", DummyClassifier(strategy="most_frequent"))]),
        "MultinomialLogistic": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))]),
        "RandomForest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestClassifier(n_estimators=250, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))]),
    }
    if HAS_LGBM:
        models["LightGBM"] = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", LGBMClassifier(n_estimators=500, learning_rate=.03, num_leaves=31, min_child_samples=50, class_weight="balanced", objective="multiclass", random_state=RANDOM_STATE, n_jobs=-1))])
    return models

## 7. Fit RL-label models

In [ ]:
# Use the same selected feature set as Project B when it exists.
MODEL_FEATURES = project_b_features if "project_b_features" in globals() else decorrelated_feature_cols
MODEL_FEATURES = [c for c in MODEL_FEATURES if c in df_train.columns and c in df_valid.columns and c in df_test.columns]
print("Model feature count:", len(MODEL_FEATURES))

rl_results = []
rl_fitted_models = {}
rl_prediction_frames = {}

REGRESSION_TARGETS = [t for t in ["target__rl_action_quality", "target__rl_reward_trade"] if t in RL_MODEL_TARGETS]
BINARY_TARGETS = [t for t in ["target__rl_trade_binary"] if t in RL_MODEL_TARGETS]
MULTICLASS_TARGETS = [t for t in ["target__rl_action_3class"] if t in RL_MODEL_TARGETS and df_train[t].dropna().nunique() >= 3]

for target_col in REGRESSION_TARGETS:
    X_train, y_train, meta_train = get_model_frame(df_train, MODEL_FEATURES, target_col)
    X_valid, y_valid, meta_valid = get_model_frame(df_valid, MODEL_FEATURES, target_col)
    X_test, y_test, meta_test = get_model_frame(df_test, MODEL_FEATURES, target_col)
    for model_name, model in get_regression_models().items():
        print("Regression:", target_col, model_name)
        model.fit(X_train, y_train)
        rl_fitted_models[(target_col, model_name)] = model
        frames = []
        for split_name, X, y, meta in [("valid", X_valid, y_valid, meta_valid), ("test", X_test, y_test, meta_test)]:
            pred = model.predict(X)
            rl_results.append({"Task":"Regression", "Target":target_col, "Model":model_name, "Split":split_name, **regression_metrics(y, pred)})
            pf = meta.copy(); pf["y_true"] = np.asarray(y); pf["score"] = pred; pf["split"] = split_name; frames.append(pf)
        rl_prediction_frames[(target_col, model_name)] = pd.concat(frames, ignore_index=True)

for target_col in BINARY_TARGETS:
    X_train, y_train, meta_train = get_model_frame(df_train, MODEL_FEATURES, target_col)
    X_valid, y_valid, meta_valid = get_model_frame(df_valid, MODEL_FEATURES, target_col)
    X_test, y_test, meta_test = get_model_frame(df_test, MODEL_FEATURES, target_col)
    y_train, y_valid, y_test = y_train.astype(int), y_valid.astype(int), y_test.astype(int)
    for model_name, model in get_binary_models(y_train).items():
        print("Binary:", target_col, model_name)
        model.fit(X_train, y_train)
        rl_fitted_models[(target_col, model_name)] = model
        frames = []
        for split_name, X, y, meta in [("valid", X_valid, y_valid, meta_valid), ("test", X_test, y_test, meta_test)]:
            score = get_positive_proba(model, X); pred = (score >= .5).astype(int)
            rl_results.append({"Task":"Binary Classification", "Target":target_col, "Model":model_name, "Split":split_name, **binary_metrics(y, pred, score)})
            pf = meta.copy(); pf["y_true"] = np.asarray(y); pf["score"] = score; pf["pred_0_5"] = pred; pf["split"] = split_name; frames.append(pf)
        rl_prediction_frames[(target_col, model_name)] = pd.concat(frames, ignore_index=True)

for target_col in MULTICLASS_TARGETS:
    X_train, y_train, meta_train = get_model_frame(df_train, MODEL_FEATURES, target_col)
    X_valid, y_valid, meta_valid = get_model_frame(df_valid, MODEL_FEATURES, target_col)
    X_test, y_test, meta_test = get_model_frame(df_test, MODEL_FEATURES, target_col)
    y_train, y_valid, y_test = y_train.astype(int), y_valid.astype(int), y_test.astype(int)
    for model_name, model in get_multiclass_models().items():
        print("Multiclass:", target_col, model_name)
        model.fit(X_train, y_train)
        rl_fitted_models[(target_col, model_name)] = model
        for split_name, X, y, meta in [("valid", X_valid, y_valid, meta_valid), ("test", X_test, y_test, meta_test)]:
            pred = model.predict(X)
            rl_results.append({"Task":"Multiclass Classification", "Target":target_col, "Model":model_name, "Split":split_name, **multiclass_metrics(y, pred)})

rl_results_df = pd.DataFrame(rl_results)
display(rl_results_df.sort_values(["Target", "Split", "Model"]))

## 8. Validation-tuned binary thresholds and calibration

In [ ]:
def tune_threshold(y_true, score, metric="f1"):
    rows = []
    for threshold in np.linspace(.01, .99, 99):
        pred = (score >= threshold).astype(int)
        value = f1_score(y_true, pred, zero_division=0) if metric == "f1" else balanced_accuracy_score(y_true, pred)
        rows.append((threshold, value))
    return max(rows, key=lambda x: x[1])

rl_thresholds = []
for (target_col, model_name), pf in rl_prediction_frames.items():
    if target_col not in BINARY_TARGETS:
        continue
    valid = pf[pf["split"] == "valid"]
    test = pf[pf["split"] == "test"].copy()
    for metric in ["f1", "balanced_accuracy"]:
        threshold, valid_value = tune_threshold(valid["y_true"].astype(int), valid["score"], metric)
        test_pred = (test["score"] >= threshold).astype(int)
        metrics = binary_metrics(test["y_true"].astype(int), test_pred, test["score"])
        rl_thresholds.append({"Target":target_col, "Model":model_name, "Threshold_metric":metric, "Threshold":threshold, "Valid_metric":valid_value, **{f"Test_{k}":v for k,v in metrics.items()}})

rl_thresholds_df = pd.DataFrame(rl_thresholds)
display(rl_thresholds_df)

In [ ]:
# Reliability curves for binary models
for (target_col, model_name), pf in rl_prediction_frames.items():
    if target_col not in BINARY_TARGETS:
        continue
    fig, ax = plt.subplots(figsize=(5, 5))
    for split_name in ["valid", "test"]:
        sub = pf[pf["split"] == split_name]
        if sub["y_true"].nunique() < 2:
            continue
        frac_pos, mean_pred = calibration_curve(sub["y_true"].astype(int), sub["score"], n_bins=10, strategy="quantile")
        ax.plot(mean_pred, frac_pos, marker="o", label=split_name)
    ax.plot([0,1],[0,1], linestyle="--", label="perfect")
    ax.set_title(f"Calibration: {target_col} / {model_name}")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed positive rate"); ax.legend()
    plt.show()

## 9. Ranking, Top-K, and subgroup diagnostics

In [ ]:
def daily_cross_sectional_diagnostics(pf, k_values=(5, 10, 20)):
    work = pf.copy()
    work["date"] = pd.to_datetime(work[TIMESTAMP_COL], errors="coerce").dt.date
    rows = []
    for date, g in work.groupby("date"):
        if len(g) < 3 or g["y_true"].nunique() < 2 or g["score"].nunique() < 2:
            continue
        row = {
            "date": date,
            "n": len(g),
            "spearman": spearmanr(g["y_true"], g["score"]).correlation,
            "pearson": pearsonr(g["y_true"], g["score"])[0],
        }
        baseline = g["y_true"].mean()
        for k in k_values:
            kk = min(k, len(g))
            top = g.nlargest(kk, "score")
            row[f"top{k}_mean_target"] = top["y_true"].mean()
            row[f"top{k}_lift"] = top["y_true"].mean() / baseline if baseline != 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def subgroup_prediction_metrics(pf, group_col, task, min_n=30):
    rows = []
    for group, g in pf.groupby(group_col, observed=True):
        if len(g) < min_n:
            continue
        row = {group_col: group, "n": len(g), "mean_target": g["y_true"].mean(), "mean_score": g["score"].mean()}
        if task == "regression":
            row.update(regression_metrics(g["y_true"], g["score"]))
        else:
            pred = (g["score"] >= .5).astype(int)
            row.update(binary_metrics(g["y_true"].astype(int), pred, g["score"]))
        rows.append(row)
    return pd.DataFrame(rows)

rl_daily_diagnostics = {}
rl_subgroup_diagnostics = {}

for key, pf in rl_prediction_frames.items():
    target_col, model_name = key
    task = "binary" if target_col in BINARY_TARGETS else "regression"
    for split_name in ["valid", "test"]:
        sub = pf[pf["split"] == split_name].copy()
        rl_daily_diagnostics[(target_col, model_name, split_name)] = daily_cross_sectional_diagnostics(sub)
        if TIMESTAMP_COL in sub.columns:
            sub["year"] = pd.to_datetime(sub[TIMESTAMP_COL], errors="coerce").dt.year
        rl_subgroup_diagnostics[(target_col, model_name, split_name, "year")] = subgroup_prediction_metrics(sub, "year", task, min_n=30)
        rl_subgroup_diagnostics[(target_col, model_name, split_name, "ticker")] = subgroup_prediction_metrics(sub, TICKER_COL, task, min_n=30)
        if INDUSTRY_COL and INDUSTRY_COL in sub.columns:
            rl_subgroup_diagnostics[(target_col, model_name, split_name, "industry")] = subgroup_prediction_metrics(sub, INDUSTRY_COL, task, min_n=30)

# Compact summary of daily ranking diagnostics
ranking_rows = []
for key, table in rl_daily_diagnostics.items():
    if table.empty: continue
    target_col, model_name, split_name = key
    row = {"Target":target_col, "Model":model_name, "Split":split_name, "days":len(table)}
    for c in [c for c in table.columns if c not in ["date", "n"]]:
        row[f"mean_{c}"] = table[c].mean()
    ranking_rows.append(row)
rl_daily_summary_df = pd.DataFrame(ranking_rows)
display(rl_daily_summary_df)

In [ ]:
# Within-ticker ranking: does the model order each ticker's own observations correctly over time?
def within_ticker_ranking(pf, min_n=30):
    rows = []
    for ticker, g in pf.groupby(TICKER_COL, observed=True):
        g = g.dropna(subset=["y_true", "score"])
        if len(g) < min_n or g["y_true"].nunique() < 2 or g["score"].nunique() < 2:
            continue
        rows.append({
            "ticker": ticker,
            "n": len(g),
            "spearman": spearmanr(g["y_true"], g["score"]).correlation,
            "pearson": pearsonr(g["y_true"], g["score"])[0],
        })
    return pd.DataFrame(rows).sort_values("spearman", ascending=False)

rl_within_ticker = {
    (*key, split): within_ticker_ranking(pf[pf["split"] == split])
    for key, pf in rl_prediction_frames.items()
    for split in ["valid", "test"]
}

## 10. Confusion matrices and selected-model inspection

In [ ]:
# Inspect the strongest non-dummy binary model according to validation PR-AUC.
if BINARY_TARGETS and not rl_results_df.empty:
    target_col = BINARY_TARGETS[0]
    candidates = rl_results_df[(rl_results_df["Target"] == target_col) & (rl_results_df["Split"] == "valid") & (~rl_results_df["Model"].str.startswith("Dummy"))]
    if not candidates.empty:
        selected_model = candidates.sort_values("PR_AUC", ascending=False).iloc[0]["Model"]
        pf = rl_prediction_frames[(target_col, selected_model)]
        for split_name in ["valid", "test"]:
            sub = pf[pf["split"] == split_name]
            pred = (sub["score"] >= .5).astype(int)
            print(f"\n{target_col} — {selected_model} — {split_name}")
            print(confusion_matrix(sub["y_true"].astype(int), pred))
            print(classification_report(sub["y_true"].astype(int), pred, zero_division=0))

## 11. Recommended interpretation and next model extensions

### Recommended core benchmark structure

- Treat **`action_quality` regression/ranking** as the main RL-label experiment. Report MAE/RMSE/R² only as supporting metrics; emphasize Pearson/Spearman, daily cross-sectional Spearman, Top-K target quality and lift.
- Treat **trade vs no-trade classification** as the main decision experiment. Report ROC-AUC, PR-AUC, Brier/calibration, validation-tuned thresholds, balanced accuracy and Top-K diagnostics.
- Use **`reward_trade` regression** as a sensitivity analysis because it embeds execution and action-cost assumptions.
- Use **three-class long/no-trade/short classification** only when the export genuinely provides all three classes in one coherent label. Do not force separate long and short evaluator outputs into one class without confirming their joint construction.

### Possible extensions after the baseline

1. **Two-stage hurdle model:** first predict trade/no-trade; then predict `action_quality` among trade-worthy observations.
2. **Learning-to-rank:** train by date groups so the objective directly matches daily stock selection.
3. **Multi-task model:** jointly predict action probability and action quality, while keeping the transparent single-task baselines.
4. **Sample weighting:** down-weight highly overlapping adjacent labels or apply recency weights only as a separately documented robustness check.
5. **Downstream simulator evaluation:** convert predicted scores into daily selections/actions and evaluate realised/simulator trading outcomes. Predictive metrics alone do not establish profitability.

## 12. Export RL-label results

In [ ]:
RL_OUTPUT_DIR = output_dir / "RLlabel_diagnostics"
RL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rl_inventory_train.to_csv(RL_OUTPUT_DIR / "rl_label_inventory_train.csv", index=False)
rl_target_summaries.to_csv(RL_OUTPUT_DIR / "rl_target_summaries.csv", index=False)
rl_results_df.to_csv(RL_OUTPUT_DIR / "rl_model_results.csv", index=False)
rl_thresholds_df.to_csv(RL_OUTPUT_DIR / "rl_binary_thresholds.csv", index=False)
rl_daily_summary_df.to_csv(RL_OUTPUT_DIR / "rl_daily_ranking_summary.csv", index=False)

for target, table in rl_target_by_year.items():
    table.to_csv(RL_OUTPUT_DIR / f"{target}_target_by_year.csv", index=False)
for target, table in rl_target_by_ticker.items():
    table.to_csv(RL_OUTPUT_DIR / f"{target}_target_by_ticker.csv", index=False)
for target, table in rl_target_by_industry.items():
    table.to_csv(RL_OUTPUT_DIR / f"{target}_target_by_industry.csv", index=False)

for key, table in rl_subgroup_diagnostics.items():
    target, model, split, group = key
    safe = lambda x: str(x).replace("/", "_").replace("\\", "_").replace(":", "_")
    table.to_csv(RL_OUTPUT_DIR / f"{safe(target)}__{safe(model)}__{split}__by_{group}.csv", index=False)

for key, table in rl_within_ticker.items():
    target, model, split = key
    safe = lambda x: str(x).replace("/", "_").replace("\\", "_").replace(":", "_")
    table.to_csv(RL_OUTPUT_DIR / f"{safe(target)}__{safe(model)}__{split}__within_ticker.csv", index=False)

print("Exported RL-label diagnostics to:", RL_OUTPUT_DIR)